In [1]:

# Imports

import helpers.helper_functions as hf
import mne
import os.path as op
from mne.channels import combine_channels
import pandas as pd
from mne.beamformer import make_lcmv, apply_lcmv_epochs
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import hilbert
import helpers.test_circ_plot as circ_plot
import gc
import helpers.stc_helper as stc_helper
import time
from pycircstat2.hypothesis import rayleigh_test
ss = hf.settings_dict()

In [3]:
for subject_index in ss['subject_idx_list']:

    # loop over each event type
    for event_id in ss['event_id_list']:

        event_name = str(event_id)
        duty_cycle = ss['event_name_list'][event_id - 1]
        subjects_dir = ss['fs_subjects_dir']
        subject = ss['subject_id_list'][subject_index]
        print("loading dataset for subject: ", subject)

        save_dir = Path(ss['hilbert_ref_dir']) / subject / event_name
        save_dir.mkdir(parents=True, exist_ok=True)

        # load hilbert stc data
        hilbert_stc_file = Path(ss['hilbert_ref_dir']) / subject / event_name / f"{subject}-event-{event_name}-hilbert-ret-ref-vol.stc"
        stc = mne.read_source_estimate(hilbert_stc_file)

        # stc.data shape: (n_voxels, n_times) — phase differences in radians [-pi, pi]
        n_voxels, n_times = stc.data.shape

        z_stats = np.zeros(n_voxels)

        # Rayleigh test to get z-scores

        for voxel_idx in range(n_voxels):
            result = rayleigh_test(stc.data[voxel_idx])    # angles in radians
            z_stats[voxel_idx] = result.z       # Z = n * r^2

        z_stc       = stc.copy().crop(0.7, 0.7 + 1/ss['fs'])
        z_stc.data  = z_stats[:, np.newaxis]   # (n_voxels, 1)

        z_stc.save(save_dir / f"{subject}-event-{event_name}-z-vol.stc" , overwrite=True)


loading dataset for subject:  0005_3SJ
delta_r min  : -0.9455
delta_r max  : 0.7083
delta_r mean : -0.2622
Voxels > 0   : 291 / 1440
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
loading dataset for subject:  0005_3SJ
delta_r min  : -0.9309
delta_r max  : 0.8099
delta_r mean : -0.3608
Voxels > 0   : 166 / 1440
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
loading dataset for subject:  0005_3SJ
delta_r min  : -0.9058
delta_r max  : 0.7076
delta_r mean : -0.3577
Voxels > 0   : 174 / 1440
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
loading dataset for subject:  0005_3SJ
delta_r min  : -0.9212
delta_r max  : 0.7587
delta_r mean : -0.2459
Voxels > 0   : 318 / 1440
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
loading dataset for subject:  0005_3SJ
delta_r min  : -0.9823
delta_r max  : 0.8210
delta_r mean : -0.2845
Voxels > 0   : 233 / 1440